In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import hydra

from genpp import BASE_DIR
from genpp.configs import register_resolvers

try:
    register_resolvers()
except ValueError:
    pass

# Test if Model works in general

In [14]:
with hydra.initialize_config_dir(version_base=None, config_dir=str(BASE_DIR / "configs")):
    cfg = hydra.compose(config_name="base_fmuvit")

In [15]:
datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup("fit")
dataloader = datamodule.train_dataloader()

Configuration hash: 1333871877b7b5de25d7b65483ae3f7608524ce480ab5274278158e8a78c286c
Cached tensor data found. Verifying configuration...
Using cached tensor data from /home/feik/GenPP/src/genpp/data/weatherbench2/cache/tensor_1333871877b7b5de25d7b65483ae3f7608524ce480ab5274278158e8a78c286c.pt.


In [16]:
cfg.model.n_samples = (
    16  # for testing purposes only (this only influences the predict step for the fm models)
)

In [17]:
model = hydra.utils.instantiate(
    cfg.model,
    rescaler=datamodule.y_reverseModules if cfg.model.use_rescaler else None,
    internal_td_scaling="learned",
)

/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'backbone' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['backbone'])`.


In [18]:
trainer = hydra.utils.instantiate(
    cfg.trainer,
    logger=None,
    detect_anomaly=True,
    accelerator="cpu",
    max_epochs=10,
    overfit_batches=10,
    check_val_every_n_epoch=5,
)
trainer.fit(model, datamodule=datamodule)

You have turned on `Trainer(detect_anomaly=True)`. This will significantly slow down compute speed and is recommended only for model debugging.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.

  | Name                | Type             | Params | Mode 
-----------------------------------------------------------------
0 | backbone            | UViT             | 88.0 K | train
1 | crop                | CropND           | 0      | train
2 | solver              | ODESolver        | 88.0 K | train
3 | internal_td_scaling | LearnedTDScaling | 97     | train
-----------------------------------------------------------------
88.1 K    Trainable params
0         Non-trainable params
88.1 K    Total params
0.352     Total estimated model params size (MB)
61        Modules in train mode
0         Modules in eval mode


/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:252: You requested to overfit but enabled train dataloader shuffling. We are turning off the train dataloader shuffling for you.


Epoch 9: 100%|██████████| 10/10 [00:02<00:00,  4.78it/s, v_num=0, train_loss=0.676, val_loss_var_0=0.536, val_loss_var_1=0.799]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 10/10 [00:02<00:00,  4.77it/s, v_num=0, train_loss=0.676, val_loss_var_0=0.536, val_loss_var_1=0.799]


## Try to load the model from checkpoint

In [19]:
import os
from pathlib import Path

from genpp.models.fm.base import FlowMatchingModel

cwd = Path(os.getcwd())
checkpoint_path = list((cwd / "lightning_logs" / "version_0" / "checkpoints").glob("*.ckpt"))[0]
checkpoint_path

PosixPath('/home/feik/GenPP/src/genpp/models/lightning_logs/version_0/checkpoints/epoch=9-val_loss=0.67.ckpt')

In [22]:
for batch in dataloader:
    break

In [20]:
mod_new = FlowMatchingModel.load_from_checkpoint(checkpoint_path)
type(mod_new.backbone)

genpp.models.fm.fm_uvit.UViT

In [23]:
mod_new._calc_loss(batch)

tensor([[[[2.5700e-01, 1.2983e-02, 3.1210e-02,  ..., 8.7195e-02,
           1.7295e-01, 7.5005e-01],
          [5.7218e-01, 1.5763e+00, 1.3672e-01,  ..., 4.5971e+00,
           6.3243e-02, 1.3749e+00],
          [6.0507e-01, 6.1821e-01, 1.7082e-01,  ..., 8.7281e-01,
           2.8498e-03, 6.5427e-01],
          ...,
          [8.2545e-01, 2.7189e-01, 4.7404e-01,  ..., 9.0563e-02,
           5.5460e-01, 8.1849e-04],
          [4.1217e-01, 5.6508e-02, 1.1953e-01,  ..., 4.5413e-02,
           2.8662e-01, 1.6511e-04],
          [2.2996e-02, 1.1146e+00, 9.4990e-02,  ..., 4.4575e-03,
           1.8623e-02, 4.2639e-01]],

         [[3.8119e-01, 8.8484e-01, 6.0320e-01,  ..., 5.7003e-03,
           2.6424e-01, 2.0711e-02],
          [2.0446e-05, 5.7584e-01, 4.3020e-01,  ..., 6.5089e+00,
           8.7202e-02, 4.1759e-01],
          [1.0702e+00, 5.6942e-01, 1.5116e-01,  ..., 3.0548e+00,
           2.0504e-01, 1.1192e-01],
          ...,
          [1.0823e-01, 2.0619e-02, 1.0745e-01,  ..., 1.0485

In [ ]:
res = mod_new.predict_step(batch)